# JP-1 G1 診断：G0 の G1 が通らなかった原因調べ（GPU・Sarashina2.2 1B）

G0 の1回目（head `2813ef1`）は、本番前の確認（E2・G1・B1）の **G1 の日本語の確認文**で止まりました。キャッシュを使う土台の道と、毎回ぜんぶ計算し直す貪欲の道が、64 トークンの途中で分かれたためです。これは凍結した計画のゲートではなく、原因を調べるためだけのノートブックです。

- 1B モデルを **bfloat16（G0 の1回目と同じ）** と **float32（方針 A）** の両方で読み込み、それぞれで次を調べます：
  - G0 のセル6 と同じ E2・G1・B1（合格の線もそのまま）
  - 何トークン目で分かれるか、そこでの上位2つの差（最大 512 トークンまで）
  - キャッシュを使う道を2回作って、毎回同じになるか
  - G0 のセル7 と同じ時間の見積もり（G0 の1回目の長さの中央値を使う）と、GPU のメモリの最大
- 使うのは無害な確認文だけです。MGSM は読みません。画面とファイルに出すのは数・真偽・ハッシュだけです。
- 不合格でも止まらずに最後まで回ります（調べるためのノートブックなので）。止まるのは版が合わないときだけです。

**使い方**：ランタイムは **GPU（L4）**。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で **15〜25 分くらい**。結果は Drive の `DECISION_POINTS/jp1/G1DIAG/`。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, GPU, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 G1診断（G0で止まったG1の原因調べ・GPU・Sarashina 1B・bf16とfp32）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 G1 diag', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_g1diag'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイム（L4）にしてね"
GPU_NAME = gpu.stdout.strip().split(",")[0]
EXPECTED_HEAD = "8b04f77886565388c5ca3ba1c87506e18547e93a"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
D = Path("/content/drive/MyDrive/DECISION_POINTS/jp1/G1DIAG")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
STARTED = time.time()
print("GPU:", GPU_NAME, "| head:", head)

In [ ]:
# Cell 2 — pinned packages and version checks (the 1B model only; MGSM is not read).  G0 run 1's median base lengths.
from pork import jp1 as P
from pork import jp1_g1diag as DG
PINS = [f"{k}=={v}" for k, v in P.PACKAGES.items()] + ["protobuf"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
CHECKS = {"packages": {k: md.version(k) for k in P.PACKAGES}, "torch": md.version("torch"),
          "gate_doc_sha256_ok": P.gate_doc_sha256(REPO) == P.GATE_DOC_SHA256}
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
SPEC = P.MODELS["1b"]
MODEL_DIR = Path("/content/sarashina_1b")
snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(MODEL_DIR), token=False,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
_tok = AutoTokenizer.from_pretrained(str(MODEL_DIR))
_vocab = sorted(_tok.get_vocab().items())
CHECKS["tokenizer_1b"] = {
    "vocab_sha256_ok": hashlib.sha256(json.dumps(_vocab, ensure_ascii=False).encode("utf-8")).hexdigest() == P.TOKENIZER_VOCAB_SHA256,
    "chat_template_sha256_ok": hashlib.sha256((_tok.chat_template or "").encode("utf-8")).hexdigest() == P.CHAT_TEMPLATE_SHA256}
del _tok, _vocab
VERSION_OK = (CHECKS["gate_doc_sha256_ok"] and CHECKS["packages"] == P.PACKAGES
              and CHECKS["torch"].split("+")[0] == P.TORCH_BASE_VERSION and all(CHECKS["tokenizer_1b"].values()))
_g0 = json.loads(Path("/content/drive/MyDrive/DECISION_POINTS/jp1/G0/G0_ACCURACY.json").read_text(encoding="utf-8"))
assert _g0["chosen"] == "1b", "G0 の1回目で選んだモデルが 1B ではありません — Claude に見せてね"
MEDIAN = {L: int(_g0["1b"][L]["median_tokens"]) for L in ("ja", "en")}
del _g0                                             # only the two medians are used
def write_json(name, obj):
    p = D / name
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=1, sort_keys=True, default=str), encoding="utf-8")
    return hashlib.sha256(p.read_bytes()).hexdigest()
write_json("G1DIAG_CHECKS.json", {"head": EXPECTED_HEAD, "checks": CHECKS, "version_ok": VERSION_OK, "gpu": GPU_NAME,
                                  "g0_run1_median_base_tokens": MEDIAN})
print(json.dumps({"version_ok": VERSION_OK, "torch": CHECKS["torch"], "median_base_tokens": MEDIAN}, indent=1))
assert VERSION_OK, P.STOP_VERSION + " — 推論の前に止めました。Claude に見せてね。"
print("VERSIONS OK")

In [ ]:
# Cell 3 — the checks, written once; cells 4 and 5 run them for bfloat16 and float32.
import gc
import torch as torch_mod
from pork import kv_cached_backend as kv
from pork import jp1_runner as rr
from pork import pork_t_v1 as pt
from pork import quality_pork_q8t as q8
from pork.ct1_runner import torch_fns
from pork.dp1_forks_runner import generate_base
from pork.ct1_forks_replacement import generate_batch
Bt = pt.backend("torch", "cuda")

def _top2(x):
    """top-1 / top-2 by logit, ties by the smaller id (as generate_base and the G1 reference)."""
    assert bool(torch_mod.isfinite(x).all()), "logits に NaN / inf が出ました — Claude に見せてね"
    m1 = x.max()
    i1 = int((x == m1).nonzero()[0, 0])
    y = x.clone(); y[i1] = float("-inf")
    m2 = y.max()
    i2 = int((y == m2).nonzero()[0, 0])
    return i1, float(m1), i2, float(m2)
OPS = {"top2": _top2, "max_abs_diff": lambda a, b: float((a - b).abs().max()), "at": lambda x, i: float(x[i])}

def precision_flags():
    out = {}
    for k, f in (("cuda_matmul_allow_tf32", lambda: bool(torch_mod.backends.cuda.matmul.allow_tf32)),
                 ("cudnn_allow_tf32", lambda: bool(torch_mod.backends.cudnn.allow_tf32)),
                 ("float32_matmul_precision", lambda: torch_mod.get_float32_matmul_precision()),
                 ("cuda_matmul_fp32_precision", lambda: str(torch_mod.backends.cuda.matmul.fp32_precision)),
                 ("cuda_matmul_allow_bf16_reduced_precision_reduction",
                  lambda: bool(torch_mod.backends.cuda.matmul.allow_bf16_reduced_precision_reduction))):
        try:
            out[k] = f()
        except Exception as e:
            out[k] = "unavailable:" + type(e).__name__
    return out

def load(dtype):
    spec = dict(P.MODELS["1b"], dtype=dtype)
    _t, tokc, model = rr.load_model(str(MODEL_DIR), None, "cuda", spec)
    assert str(model.dtype) == "torch." + dtype and model.config._attn_implementation == spec["attn_implementation"]
    return tokc, model, kv.eos_ids(tokc, model)

def gate_replica(MODEL, TOKC, EOS):
    """G0 cell 6 (E2 / G1 / B1 on the harmless prompts), unchanged except that it is a function."""
    single, batch = torch_fns(MODEL, torch_mod, "cuda")
    e2, g1 = [], []
    for name, text in P.CHECK_PROMPTS:
        ids = kv.chat_prompt_ids(TOKC, text)
        step = single()
        seq, past, new, worst, agree = list(ids), None, tuple(ids), 0.0, True
        for _ in range(P.E2_STEPS):
            a, past = step(new, past)
            with torch_mod.no_grad():
                bfull = MODEL(input_ids=torch_mod.tensor([seq], device="cuda"), use_cache=False).logits[0, -1].float()
            worst = max(worst, float((a - bfull).abs().max()))
            ta, tb = int(a.argmax()), int(bfull.argmax())
            agree = agree and ta == tb
            seq.append(tb); new = (tb,)
        e2.append({"prompt": name, "max_abs_logit_diff": worst, "argmax_agree": agree})
        ref_tokens, seq = [], list(ids)
        for _ in range(P.G1_STEPS):
            with torch_mod.no_grad():
                tok_ = q8._greedy([float(v) for v in MODEL(input_ids=torch_mod.tensor([seq], device="cuda"),
                                                            use_cache=False).logits[0, -1].float().tolist()])
            ref_tokens.append(tok_); seq.append(tok_)
            if tok_ in set(EOS):
                break
        base = generate_base(Bt, single(), ids, EOS, len(ref_tokens))["token_ids"]
        g1.append({"prompt": name, "steps": len(ref_tokens), "identical": base == ref_tokens})
    ids = kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1])
    _bs, _ss = batch(), single()
    bp, sp, bn, sn = None, None, [tuple(ids)] * P.K, tuple(ids)
    b1_agree = 0
    for _ in range(32):
        bl, bp = _bs(bn, bp); sl, sp = _ss(sn, sp)
        t_ = int(sl.argmax()); b1_agree += int(int(bl[0].argmax()) == t_)
        bn, sn = [(t_,)] * P.K, (t_,)
    del _bs, _ss, bp, sp, bl, sl
    g1.append({"prompt": "B1_batched_vs_single", "steps": 32, "identical": b1_agree >= P.B1_MIN_AGREE, "agree_steps": b1_agree})
    return {"E2": e2, "G1_B1": g1, "pass": all(r["argmax_agree"] for r in e2) and all(r["identical"] for r in g1)}

def diagnose(MODEL, TOKC, EOS):
    """Per-step look along the full-replay greedy path (up to MAX_NEW_TOKENS), and the free-running cached greedy x2."""
    single, _ = torch_fns(MODEL, torch_mod, "cuda")
    def full(seq):
        with torch_mod.no_grad():
            return MODEL(input_ids=torch_mod.tensor([list(seq)], device="cuda"), use_cache=False).logits[0, -1].float()
    out = {}
    for name, text in P.CHECK_PROMPTS:
        ids = kv.chat_prompt_ids(TOKC, text)
        tf = DG.teacher_forced(single(), full, ids, EOS, P.MAX_NEW_TOKENS, OPS)
        runs = [generate_base(Bt, single(), ids, EOS, len(tf["path"]))["token_ids"] for _ in range(2)]
        out[name] = {"teacher_forced": DG.summarize(tf["rows"]), "free_run": DG.free_run_check(tf["path"], runs, tf["rows"])}
        del tf, runs
    return out

def timing(MODEL, TOKC):
    """G0 cell 7's timing smoke (same formula and stop lines), plus the peak GPU memory of one worker at the longest
    prefix (prompt + MAX_NEW_TOKENS tokens, 32 rows)."""
    single, batch = torch_fns(MODEL, torch_mod, "cuda")
    total_gb = torch_mod.cuda.get_device_properties(0).total_memory / 1e9
    WORKERS = max(1, min(rr.MAX_WORKERS, kv.physical_cores(), int(total_gb // 10)))
    ids = kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1])
    no_eos = (-1,)
    generate_base(Bt, single(), ids, no_eos, 8)
    n = P.TIMING_SMOKE_STEPS
    per = {}
    t0 = time.time(); g = generate_base(Bt, single(), ids, no_eos, n); per["BASE"] = (time.time() - t0) / n
    streams = [P.stream("free", 0, 0, j) for j in range(P.K)]
    generate_batch(Bt, batch(), ids, streams, no_eos, P.TEMPERATURE, 4)
    t0 = time.time(); generate_batch(Bt, batch(), ids, streams, no_eos, P.TEMPERATURE, n); per["BATCH32"] = (time.time() - t0) / n
    long_ids = tuple(ids) + tuple(g["token_ids"]) * 12
    t0 = time.time(); generate_batch(Bt, batch(), long_ids, streams, no_eos, P.TEMPERATURE, 1); per["PREFILL"] = time.time() - t0
    KPP = 3 + 1 / P.FREE_EVERY
    def per_item(Lb):
        rest = [Lb - t for t in range(0, Lb, P.STRIDE)]
        return Lb * per["BASE"] + sum(rest) * KPP * per["BATCH32"] + len(rest) * KPP * per["PREFILL"]
    NB = {}
    for L in ("ja", "en"):
        hours = len(P.HALVES["A"]) * per_item(MEDIAN[L]) / WORKERS / 3600
        NB[L] = {"median_base_tokens": MEDIAN[L], "typical_hours_per_notebook": hours,
                 "calibrated_hours_per_notebook": hours * P.TIMING_FACTOR}
    main_cu = 2 * sum(v["calibrated_hours_per_notebook"] for v in NB.values()) * P.L4_CU_PER_HOUR
    # peak memory of one worker: 32 rows, prefix = the check prompt + 512 tokens, and a safety length of 1,024 tokens
    # (the MGSM prompts are longer than the check prompt; eager attention's temporaries grow with the square).
    mem = {}
    for label, length in (("prompt_plus_512", len(ids) + P.MAX_NEW_TOKENS), ("tokens_1024", 1024)):
        gc.collect(); torch_mod.cuda.empty_cache(); torch_mod.cuda.reset_peak_memory_stats()
        pre = (tuple(ids) + tuple(g["token_ids"]) * (length // len(g["token_ids"]) + 1))[:length]
        generate_batch(Bt, batch(), pre, streams, no_eos, P.TEMPERATURE, 2)
        free_b, total_b = torch_mod.cuda.mem_get_info()
        mem[label] = {"prefix_tokens": len(pre), "peak_allocated_gb": torch_mod.cuda.max_memory_allocated() / 1e9,
                      "peak_reserved_gb": torch_mod.cuda.max_memory_reserved() / 1e9,
                      "device_used_gb_after": (total_b - free_b) / 1e9}
    cfg = {k: getattr(MODEL.config, k, None) for k in ("num_hidden_layers", "num_attention_heads", "num_key_value_heads",
                                                       "hidden_size", "intermediate_size", "vocab_size")}
    return {"workers_g0_rule": WORKERS, "per_step_s": per, "per_notebook": NB,
            "main_run_cu_4_notebooks": main_cu,          # the 4 main notebooks only; G0 time is not included
            "per_notebook_ok": all(v["calibrated_hours_per_notebook"] <= P.TIMING_STOP_HOURS for v in NB.values()),
            "gpu_total_gb": total_gb, "memory_one_worker": mem, "model_config": cfg,
            "weights_gb": sum(p.numel() * p.element_size() for p in MODEL.parameters()) / 1e9}

def run_dtype(dtype):
    t0 = time.time()
    TOKC, MODEL, EOS = load(dtype)
    res = {"dtype": dtype, "precision_flags": precision_flags()}
    res["gate"] = gate_replica(MODEL, TOKC, EOS)
    res["diag"] = diagnose(MODEL, TOKC, EOS)
    res["timing"] = timing(MODEL, TOKC)
    res["seconds"] = time.time() - t0
    del MODEL; gc.collect(); torch_mod.cuda.empty_cache()
    return res

def short_view(res):
    d = res["diag"]
    return {"dtype": res["dtype"], "gate_pass": res["gate"]["pass"],
            "G1_identical": {r["prompt"]: r["identical"] for r in res["gate"]["G1_B1"]},
            "first_disagree_step": {k: v["teacher_forced"]["first_disagree_step"] for k, v in d.items()},
            "disagree_steps": {k: (v["teacher_forced"]["disagree_steps"], v["teacher_forced"]["steps"]) for k, v in d.items()},
            "free_run_identical": {k: v["free_run"]["identical_to_reference"] for k, v in d.items()},
            "repeat_identical": {k: v["free_run"]["repeat_identical"] for k, v in d.items()},
            "calibrated_hours_per_notebook": {L: round(v["calibrated_hours_per_notebook"], 2) for L, v in res["timing"]["per_notebook"].items()},
            "main_run_cu": round(res["timing"]["main_run_cu_4_notebooks"], 1),
            "peak_reserved_gb": {k: round(v["peak_reserved_gb"], 2) for k, v in res["timing"]["memory_one_worker"].items()},
            "seconds": round(res["seconds"])}
print("ready")

In [ ]:
# Cell 4 — bfloat16 (as G0 run 1).
RES = {}
RES["bf16"] = run_dtype("bfloat16")
write_json("G1DIAG_BF16.json", RES["bf16"])
print(json.dumps(short_view(RES["bf16"]), indent=1))

In [ ]:
# Cell 5 — float32 (direction A).  TF32 off (the PyTorch default; turned off here if the runtime had it on).
F32_FLAGS_BEFORE = precision_flags()
if F32_FLAGS_BEFORE.get("cuda_matmul_fp32_precision") == "tf32":            # the newer switch (torch 2.9 and later)
    torch_mod.backends.cuda.matmul.fp32_precision = "ieee"
elif F32_FLAGS_BEFORE.get("cuda_matmul_allow_tf32") is True or F32_FLAGS_BEFORE.get("float32_matmul_precision") in ("high", "medium"):
    torch_mod.backends.cuda.matmul.allow_tf32 = False
    torch_mod.set_float32_matmul_precision("highest")
RES["fp32"] = run_dtype("float32")
RES["fp32"]["precision_flags_before"] = F32_FLAGS_BEFORE
write_json("G1DIAG_FP32.json", RES["fp32"])
print(json.dumps(short_view(RES["fp32"]), indent=1))

In [ ]:
# Cell 6 — summary (numbers only, under 20 KB), then notify and disconnect.
SUMMARY = {"schema": "jp1_g1diag_v1", "head": EXPECTED_HEAD, "gpu": GPU_NAME, "checks": CHECKS,
           "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds"),
           "elapsed_hours": (time.time() - STARTED) / 3600, "g0_run1_median_base_tokens": MEDIAN,
           "short": {k: short_view(v) for k, v in RES.items()}, "bf16": RES["bf16"], "fp32": RES["fp32"]}
_s = json.dumps(SUMMARY, ensure_ascii=False, indent=1, sort_keys=True, default=str)     # as write_json writes it
if len(_s.encode("utf-8")) > 20000:
    SUMMARY = {k: v for k, v in SUMMARY.items() if k not in ("bf16", "fp32")}
    SUMMARY["details"] = "G1DIAG_BF16.json / G1DIAG_FP32.json"
sha_ = write_json("G1DIAG_SUMMARY.json", SUMMARY)
print("G1DIAG_SUMMARY.json sha256:", sha_)
print(json.dumps(SUMMARY["short"], indent=1))
notify("G1 診断 終わりました（結果は Drive の jp1/G1DIAG）", tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()